# Session 6: Introduction to APIs with Django REST Framework

This notebook introduces APIs, Django REST Framework (DRF), and implementing `GET` and `POST` methods using `APIView`. We extend the **Personal Task Manager** project by adding API endpoints to list and create tasks. We'll test the APIs using Postman and explore Swagger for documentation.

**Topics**:
- What are APIs and why use DRF?
- Setting up DRF and creating a serializer.
- Implementing `GET` and `POST` methods with `APIView`.
- Testing APIs with Postman.
- Introduction to Swagger and using DRF's Swagger UI.
- Exercise: Create and test API endpoints for tasks.

## Step 1: Introduction to APIs

**What is an API?**
- API (Application Programming Interface) allows different systems to communicate by sending and receiving data, typically in JSON or XML format.
- In Django, DRF simplifies building RESTful APIs, providing tools for serialization, request handling, and documentation.

**Why DRF?**
- Simplifies API development with serializers, views, and authentication.
- Provides built-in tools for testing and documentation (e.g., Swagger).

**Action**: Install DRF and add it to the project.

In [ ]:
# Install Django REST Framework
!pip install djangorestframework

## Step 2: Configure DRF

Add DRF to `INSTALLED_APPS` in the project settings.

**Action**: Update `data_dashboard/settings.py`.

In [ ]:
%%writefile data_dashboard/settings.py
# Django project settings
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    'dashboard',
    'rest_framework',  # Add DRF to installed apps
]

# Static files configuration
STATIC_URL = '/static/'
STATICFILES_DIRS = ['static']

# DRF settings
REST_FRAMEWORK = {
    'DEFAULT_PERMISSION_CLASSES': [
        'rest_framework.permissions.AllowAny',  # Allow open access for this session
    ],
}

# Other settings (e.g., DATABASES, TEMPLATES) remain unchanged
DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.sqlite3',
        'NAME': 'db.sqlite3',
    }
}
TEMPLATES = [
    {
        'BACKEND': 'django.template.backends.django.DjangoTemplates',
        'DIRS': ['templates'],
        'APP_DIRS': True,
        'OPTIONS': {
            'context_processors': [
                'django.template.context_processors.debug',
                'django.template.context_processors.request',
                'django.contrib.auth.context_processors.auth',
                'django.contrib.messages.context_processors.messages',
            ],
        },
    },
]
# Add other necessary settings as needed
SECRET_KEY = 'your-secret-key'
DEBUG = True
ALLOWED_HOSTS = []


## Step 3: Create a Serializer

Serializers convert Django models to JSON and validate incoming data. We'll create a `TaskSerializer` for the `Task` model.

**Action**: Create `dashboard/serializers.py`.

In [ ]:
%%writefile dashboard/serializers.py
# Import DRF serializer and project models
from rest_framework import serializers
from .models import Task

# Serializer for Task model
class TaskSerializer(serializers.ModelSerializer):
    class Meta:
        model = Task
        fields = ['id', 'title', 'description', 'due_date', 'priority', 'category', 'is_completed']  # Include all relevant fields


## Step 4: Implement API Views

Use `APIView` to create endpoints for listing tasks (`GET`) and creating tasks (`POST`).

**Action**: Update `dashboard/views.py` to include API views.

In [ ]:
%%writefile dashboard/views.py
# Import Django and DRF utilities
from django.shortcuts import render, redirect
from .models import Task
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from .serializers import TaskSerializer


# API view for listing and creating tasks
class TaskListCreateAPIView(APIView):
    def get(self, request):
        # Retrieve all tasks and serialize them
        tasks = Task.objects.all()
        serializer = TaskSerializer(tasks, many=True)
        return Response(serializer.data, status=status.HTTP_200_OK)

    def post(self, request):
        # Create task from POST data
        serializer = TaskSerializer(data=request.data)
        if serializer.is_valid():
            serializer.save()
            return Response(serializer.data, status=status.HTTP_201_CREATED)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

## Step 5: Update URLs

Add URL patterns for the API endpoints.

**Action**: Update `dashboard/urls.py`.

In [ ]:
%%writefile dashboard/urls.py
# Import path for URL routing
from django.urls import path
from . import views

# Define app namespace
app_name = 'dashboard'

# URL patterns for web and API endpoints
urlpatterns = [
    path('api', views.TaskListCreateAPIView.as_view(), name='task_api_list_create'),  # API for tasks
]

## Step 6: Testing APIs with Postman

**What is Postman?**
- Postman is a tool for testing APIs by sending HTTP requests (e.g., GET, POST) and viewing responses.

**Action**: Test the `GET` and `POST` endpoints using Postman.

**Instructions**:
1. Start the Django server:
   ```bash
   python manage.py runserver
   ```
2. Open Postman and create two requests:
   - **GET Request**:
     - URL: `http://127.0.0.1:8000/api/tasks/`
     - Method: GET
     - Expected Response: JSON list of tasks (e.g., `[{id: 1, title: "Write Report", ...}, ...]`)
   - **POST Request**:
     - URL: `http://127.0.0.1:8000/api/tasks/`
     - Method: POST
     - Body (JSON):
       ```json
       {
           "title": "API Task",
           "description": "Task created via API",
           "due_date": "2025-07-30",
           "priority": "medium",
           "category": 1,
           "is_completed": false
       }
       ```
     - Ensure a `TaskCategory` with `id=1` exists (add via Admin if needed).
     - Expected Response: JSON of the created task with status 201, or error with status 400 if invalid.
3. Verify the task appears in the task list (`http://127.0.0.1:8000/`) or Admin panel.

In [ ]:
# Start Django development server
!python manage.py runserver

## Step 7: Introduction to Swagger

**What is Swagger?**
- Swagger is a tool for documenting and testing APIs, providing an interactive UI to explore endpoints, methods, and schemas.
- DRF integrates with Swagger via packages like `drf-yasg` to auto-generate API documentation.

**Action**: Install `drf-yasg` and configure Swagger UI.

**Instructions**:
1. Install `drf-yasg`:
   ```bash
   pip install drf-yasg
   ```
2. Add `drf_yasg` to `INSTALLED_APPS` in `data_dashboard/settings.py`:
   ```python
   INSTALLED_APPS = [
       ...,
       'drf_yasg',
   ]
   ```
3. Update project URLs to include Swagger UI.

In [ ]:
%%writefile data_dashboard/urls.py
# Import Django and DRF-yasg utilities
from django.contrib import admin
from django.urls import path, include
from rest_framework import permissions
from drf_yasg.views import get_schema_view
from drf_yasg import openapi

# Configure Swagger schema
schema_view = get_schema_view(
    openapi.Info(
        title="Task Manager API",
        default_version='v1',
        description="API for Personal Task Manager",
    ),
    public=True,
    permission_classes=(permissions.AllowAny,),
)

# Project URL patterns
urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('dashboard.urls')),  # Include dashboard app URLs
    path('swagger/', schema_view.with_ui('swagger', cache_timeout=0), name='schema-swagger-ui'),  # Swagger UI
]

## Step 8: Using Swagger

**How to Use Swagger**:
1. After configuring `drf-yasg`, start the server (`python manage.py runserver`).
2. Open `http://127.0.0.1:8000/swagger/` in a browser.
3. Explore the API documentation:
   - See the `/api/tasks/` endpoint with `GET` and `POST` methods.
   - Click `GET /api/tasks/` to test listing tasks (execute and view JSON response).
   - Click `POST /api/tasks/`, enter JSON data (same as Postman example), and execute to create a task.
4. Check for errors (e.g., invalid `due_date` or missing `category`) in the Swagger UI response.

**Benefits**:
- Interactive UI for testing without external tools like Postman.
- Auto-generated documentation for API endpoints and schemas.

## Step 9: Practical Exercise

### Task: Create and Test API Endpoints

1. Set up the API for listing and creating tasks (done).
2. Test the API using Postman:
   - Send a `GET` request to list tasks.
   - Send a `POST` request to create a task.
3. Test the API using Swagger UI and verify the results.

**Action**: Ensure a `TaskCategory` exists and test the API.

In [ ]:
# Ensure a TaskCategory exists via Admin or shell
# Run in Django shell: python manage.py shell
from dashboard.models import TaskCategory

# Create a category if none exists
TaskCategory.objects.get_or_create(name='Work')

# Start the server for testing
!python manage.py runserver

### Instructions for Testing the Exercise
1. **Ensure Setup**:
   - Add a `TaskCategory` (e.g., `Work`, ID=1) via `http://127.0.0.1:8000/admin/` or the shell command above.
2. **Test with Postman**:
   - **GET**: Send a GET request to `http://127.0.0.1:8000/api/tasks/` to retrieve tasks.
   - **POST**: Send a POST request to `http://127.0.0.1:8000/api/tasks/` with JSON:
     ```json
     {
         "title": "API Task",
         "description": "Task created via API",
         "due_date": "2025-07-30",
         "priority": "medium",
         "category": 1,
         "is_completed": false
     }
     ```
     - Test invalid data (e.g., `title`: `Hi`) to see error responses.
3. **Test with Swagger**:
   - Open `http://127.0.0.1:8000/swagger/`.
   - Use the UI to send `GET` and `POST` requests to `/api/tasks/`.
4. **Verify Results**:
   - Check the task list at `http://127.0.0.1:8000/` or Admin panel to confirm new tasks.
   - Use the shell to query tasks.

## Step 10: Verify Data in the Shell

Use the Django shell to confirm API-created tasks are saved correctly.

**Action**: Query tasks in the shell.

In [ ]:
# Run in Django shell: python manage.py shell
# Import Task model for querying
from dashboard.models import Task

# Query all tasks
Task.objects.all()

# Query tasks created via API
Task.objects.filter(title='API Task')

## Exercise Summary

You have:
1. Installed and configured Django REST Framework.
2. Created a `TaskSerializer` for the `Task` model.
3. Implemented `TaskListCreateAPIView` for `GET` and `POST` methods.
4. Tested API endpoints using Postman.
5. Set up Swagger UI for API documentation and testing.
6. Verified API functionality via Postman, Swagger, and the Django shell.

**Test the Application**:
- Use Postman to send `GET` and `POST` requests to `http://127.0.0.1:8000/api/tasks/`.
- Explore and test APIs at `http://127.0.0.1:8000/swagger/`.
- Verify tasks in the Admin panel (`http://127.0.0.1:8000/admin/`) or task list.